# 00 - Workspace, storage, and synthetic source data

Prerequisites / execution context: Databricks Python notebook; UC compute; existing writable catalog. Change CATALOG if workspace is not your catalog. Creates three dedicated schemas, a volume and lab tables. Re-running resets the main Bronze table.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R05: Free Edition limitations](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations); [R08: Unity Catalog volumes](https://docs.databricks.com/aws/en/volumes/); [R58: UC privileges reference](https://docs.databricks.com/aws/en/data-governance/unity-catalog/access-control/privileges-reference); [R71: Import/export notebooks](https://docs.databricks.com/aws/en/notebooks/notebook-export-import).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Create objects one statement at a time
Predict which permission each statement needs. The notebook uses an existing catalog; it does not require CREATE CATALOG.


In [ ]:
for schema in (BRONZE, SILVER, GOLD):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE}.landing")
print(BRONZE, SILVER, GOLD, VOLUME)


## 2. Generate a deliberate data-quality challenge
Eight events include an exact duplicate, a newer correction, one unknown customer, a negative amount, and a malformed amount. Order IDs and customer IDs are strings to retain leading zeroes. Synthetic money uses strings in Bronze and DECIMAL in Silver. Official Databricks documentation: [R06: Medallion architecture](https://docs.databricks.com/aws/en/lakehouse/medallion); [R19: PySpark basics](https://docs.databricks.com/aws/en/pyspark/basics).


In [ ]:
import json
rows = [
 ("e1","o1","00035","100.00","2026-11-01 10:00:00",1,["web","priority"]),
 ("e1","o1","00035","100.00","2026-11-01 10:00:00",1,["web","priority"]),
 ("e2","o2","00037","50.00","2026-11-01 11:00:00",1,["store"]),
 ("e3","o1","00035","120.00","2026-11-02 10:00:00",2,["web"]),
 ("e4","o3",None,"20.00","2026-11-02 12:00:00",1,[]),
 ("e5","o4","00038","-5.00","2026-11-02 13:00:00",1,["web"]),
 ("e6","o5","00038","oops","2026-11-02 14:00:00",1,["mobile"]),
 ("e7","o6","00999","30.00","2026-11-02 15:00:00",1,["mobile"])
]
cols = ["event_id","order_id","customer_id","amount","event_time","seq","tags"]
payload = [dict(zip(cols,r)) for r in rows]
dbutils.fs.put(f"{VOLUME}/seed/orders.json", "\n".join(json.dumps(r) for r in payload), True)
raw_schema = "event_id STRING, order_id STRING, customer_id STRING, amount STRING, event_time STRING, seq LONG, tags ARRAY<STRING>"
raw = spark.read.schema(raw_schema).json(f"{VOLUME}/seed")
raw.write.format("delta").mode("overwrite").saveAsTable(f"{BRONZE}.orders_raw")
customers = spark.createDataFrame([("00035","IT"),("00037","FR"),("00038","IT")], "customer_id STRING, country STRING")
customers.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.customers")
assert raw.count()==8
assert customers.count()==3
raw.show(truncate=False)


## Verify and explain
Open Catalog Explorer. Locate the three schemas, managed Delta table and volume. Explain why the JSON file belongs in a volume and the queryable records belong in a table. Inspect table details and history.


In [ ]:
spark.sql(f"DESCRIBE DETAIL {BRONZE}.orders_raw").show(truncate=False)
spark.sql(f"DESCRIBE HISTORY {BRONZE}.orders_raw").show(truncate=False)


## Exercise
Without changing the source, state how many unique event IDs and how many unique order IDs exist. Solution: seven event IDs; six order IDs. Re-run the setup only when you intend to reset the main dataset. Ingestion notebooks use separate locations.
